# (old) Шаг 3. Файлы силового поля для остатка
*Путь словарей и тримеров (запись прежнего способа, сделан из общего `3_edd_topology.ipynb`, он в
`archive/notebooks/`). Новый путь: `pt.write_forcefield` в `2_forcefield.ipynb` - типы по окружению атома,
явные параметры только там, где их нет в поле, запись в поле без дубликатов, проверка в GROMACS.*

- **Вход**: `_mod.itp` шага 2, `data/general_atoms.json` (атомы родителя, шаг 4), заряды
  `data/AI_chrges_<мономер>.json` (Espaloma, шаг 5) или `data/RESP_chrges_<мономер>.json` (шаг 6).
- **Выход** - `force_field_files/`: `<папка>.rtp`, `residuetypes.dat`, `aminoacids.hdb`, `atomtypes.atp`.

Отличия от старого общего ноутбука:
- файлы пишутся только в `force_field_files/` этой папки и каждый раз строятся из базового поля, повторный
  запуск ничего не дублирует. Старый ноутбук дописывал строки в начало `amber14sb_mod.ff/residuetypes.dat`
  и `aminoacids.hdb` при каждом запуске; перенос в общее поле - `2_forcefield.ipynb`;
- родитель - из параметров (`parent_residue`), а не `'LYS'`;
- заряды читаются с проверкой порядка атомов (`pt.load_charges_json`), а не по индексу вслепую;
- заряды пишутся с 6 знаками (с 4 сумма расходилась с целым на ~0.0005);
- исправлена ошибка выбора «канонических» связей, углов и диэдралов: `(a and b) in список` проверяло только
  последний атом, теперь проверяются все (`all(...)`);
- `Makefile.am/.in` не пишутся: GROMACS их не читает, а старые ссылались на несуществующий `<PTM>_raw.rtp`;
  r2b в старом ноутбуке был закомментирован.
- Ядро - окружение с parmed (`topmol2`).

In [1]:
import sys
import os
import json
ROOT_DIR = globals().get('ROOT_DIR') or os.path.abspath('..')  # notebooks_and_examples/
sys.path.append(ROOT_DIR)
from parmed import gromacs
import ResParTools as pt

BASE_FF = f'{ROOT_DIR}/amber14sb_parmbsc1_cufix.ff'

In [2]:
# Параметры модификации
PTM_folder = 'Lysine_3M'      # папка модификации: имя rtp-файла в force_field_files/
PTM_name = 'Lysine_3M'        # вход шага 2: {PTM_name}_rn_H_3D
base_name = 'K3M'             # имя остатка в силовом поле
parent_residue = 'K'          # родительский остаток (однобуквенный код)
mon_name = 'Lysine_3M'        # мономер: заряды data/AI_chrges_{mon_name}.json
charges_file = f'data/AI_chrges_{mon_name}.json'   # или f'data/RESP_chrges_{mon_name}.json'
# атомы родителя, которые НЕ наследуют его тип (старое правило; новый путь определяет их по окружению):
# у лизинов HE1/HE2 - водороды при NZ, в лизине у них тип HP (рядом с N+)
exclusion_atoms = ['HE1', 'HE2']

## Топология шага 2 без протонов HW
Протоны `HW` нужны были только acpype (оборванные валентности N и C); в остатке их нет.

In [3]:
os.chdir(f'{ROOT_DIR}/{PTM_folder}')
acpype_name = f'{PTM_name}_rn_H_3D'
top_L = gromacs.GromacsTopologyFile(f'Acpype_data/{acpype_name}.acpype/{acpype_name}_mod.itp')
pt.remove_extra_H(top_L, extra_pattern_H='HW')
print('атомов:', len(top_L.atoms))

Найденные атомы для удаления: [<Atom HW1 [31]; In K3M 0>, <Atom HW2 [32]; In K3M 0>]
атомов: 31


## Типы атомов: общие с родителем атомы получают тип родителя
*Важно: атом может быть каноническим, но его валентность могла поменяться, следи за этим!
(следим на шаге сопоставления и переименовки атомов; атомы с изменённым окружением - в `exclusion_atoms`).*

Таблица: имя атома | тип acpype (itp) | тип в родителе (rtp) | итоговый тип.

In [4]:
parent_name = pt.AMINO_ACIDS[parent_residue][0].upper()          # 'LYS', 'CYS'...
parent_types = {n: t for n, t, *_ in pt.read_rtp(f'{BASE_FF}/aminoacids.rtp')[1][parent_name]['atoms']}
general_atoms = json.load(open('data/general_atoms.json'))
canonical_atoms = [name for name in general_atoms if name not in exclusion_atoms]
for atom in top_L.atoms:
    itp_type = str(atom.atom_type)
    if atom.name in canonical_atoms:
        atom.atom_type = parent_types[atom.name]
    print(f"{atom.name:5s} {itp_type:4s} {parent_types.get(atom.name, '-'):4s} {str(atom.atom_type)}"
          + ('   <- тип родителя' if str(atom.atom_type) != itp_type else ''))

N     NT   N    N   <- тип родителя
H     H    H    H
CA    CT   CX   CX   <- тип родителя
HA    H1   H1   H1
CB    CT   C8   C8   <- тип родителя
HB1   HC   HC   HC
HB2   HC   HC   HC
CG    CT   C8   C8   <- тип родителя
HG1   HC   HC   HC
HG2   HC   HC   HC
CD    CT   C8   C8   <- тип родителя
HD1   HC   HC   HC
HD2   HC   HC   HC
CE    CT   C8   C8   <- тип родителя
HE1   HP   HP   HP
HE2   HP   HP   HP
NZ    N3   N3   N3
CH1   CT   -    CT
HH11  HP   -    HP
HH12  HP   -    HP
HH13  HP   -    HP
CH2   CT   -    CT
HH21  HP   -    HP
HH22  HP   -    HP
HH23  HP   -    HP
CH3   CT   -    CT
HH31  HP   -    HP
HH32  HP   -    HP
HH33  HP   -    HP
C     C    C    C
O     O    O    O


In [5]:
# типы, которых нет в atomtypes.atp поля, дописываются в force_field_files/atomtypes.atp
pt.check_atomtypes(top_L, path_to_atp=f'{BASE_FF}/atomtypes.atp', param_folder='force_field_files')

Все используемые типы атомов указаны в /home/nikolay/Documents/VS_code/ResParTools/notebooks_and_examples/amber14sb_parmbsc1_cufix.ff/atomtypes.atp


## Заряды
Порядок атомов в файле зарядов сверяется с itp по именам. В старом формате (просто список, без имён)
проверяется только число атомов - с предупреждением.

In [6]:
charges = pt.load_charges_json(charges_file, [atom.name for atom in top_L.atoms])
print(f'сумма зарядов: {sum(charges):.4f}')

сумма зарядов: 1.0000


## rtp
Связи, углы и диэдралы, все атомы которых - канонические (общие с родителем), пишутся без параметров:
GROMACS берёт их из поля по типам атомов. Остальные - с параметрами acpype (пересчёт в единицы GROMACS).

In [7]:
def make_rtp_for_aminoacid(g_parm, charges, name, shift=0, canonical_atoms=()):
    canonical = lambda *atoms: all(a.name in canonical_atoms for a in atoms)
    rtp_text = ['[ bondedtypes ]\n',
                '; bonds  angles  dihedrals  impropers all_dihedrals nrexcl HH14 RemoveDih\n',
                '     1       1          9          4        1         3      1     0 \n', '\n',
                f'[ {name} ]\n', ' [ atoms ]\n']
    for i, atom in enumerate(g_parm.atoms):
        rtp_text.append("%6s%6s%12.6f%4d\n" % (atom.name, str(atom.atom_type), charges[i], atom.idx + shift))
    rtp_text.append(' [ bonds ]\n')
    for bond in g_parm.bonds:
        if canonical(bond.atom1, bond.atom2):
            rtp_text.append("%6s%6s\n" % (bond.atom1.name, bond.atom2.name))
        else:
            rtp_text.append("%6s%6s%12.4f%12.1f\n" % (bond.atom1.name, bond.atom2.name, 0.1 * bond.type.req,
                                                     bond.type.k * 2 * 4.184 * 100))
    rtp_text.append('   -C    N\n')
    rtp_text.append(' [ angles ]\n')
    for angle in g_parm.angles:
        names = (angle.atom1.name, angle.atom2.name, angle.atom3.name)
        if canonical(angle.atom1, angle.atom2, angle.atom3):
            rtp_text.append("%6s%6s%6s\n" % names)
        else:
            rtp_text.append("%6s%6s%6s %12.4f %8.1f\n" % (*names, angle.type.theteq, angle.type.k * 2 * 4.184))
    for funct, section in ((9, 'dihedrals'), (4, 'impropers')):
        rtp_text.append(f' [ {section} ]\n')
        for d in g_parm.dihedrals:
            if d.funct != funct:
                continue
            names = (d.atom1.name, d.atom2.name, d.atom3.name, d.atom4.name)
            if canonical(d.atom1, d.atom2, d.atom3, d.atom4):
                rtp_text.append("%6s%6s%6s%6s\n" % names)
            else:
                for t in (d.type if funct == 9 else [d.type]):
                    rtp_text.append("%6s%6s%6s%6s %8.3f %8.4f %5i\n" % (*names, t.phase, t.phi_k * 4.184, t.per))
    rtp_text.extend(['    -C    CA     N     H\n', '    CA    +N     C     O\n'])
    return ''.join(rtp_text)


rtp = make_rtp_for_aminoacid(top_L, charges, base_name, shift=1, canonical_atoms=canonical_atoms)
with open(f'force_field_files/{PTM_folder}.rtp', 'w') as f:
    f.write(rtp)
print(f'force_field_files/{PTM_folder}.rtp: {rtp.count(chr(10))} строк')

force_field_files/Lysine_3M.rtp: 208 строк


## residuetypes.dat и hdb
Копии файлов базового поля со строкой нового остатка (строка остатка заменяется, а не дописывается).
hdb (правила достройки водородов) - `pt.hdb_generator` по остатку шага 4.

In [8]:
lines = [l for l in open(f'{BASE_FF}/residuetypes.dat') if l.split()[:1] != [base_name]]
with open('force_field_files/residuetypes.dat', 'w') as f:
    f.writelines([f'{base_name}     Protein\n'] + lines)

residue = pt.pdb_to_chem(f'molecules/substructure/4_{mon_name}.pdb')[f'4_{mon_name}']
add_hdb = pt.hdb_generator(residue, base_name)
print(*add_hdb, sep='')
with open('force_field_files/aminoacids.hdb', 'w') as f:
    f.writelines(add_hdb + open(f'{BASE_FF}/aminoacids.hdb').readlines())

K3M	9
1	1	H	N	-C	CA	
1	5	HA	CA	N	CB	C
2	6	HB	CB	CA	CG
2	6	HG	CG	CB	CD
2	6	HD	CD	CG	CE
2	6	HE	CE	CD	NZ
3	4	HH1	CH1	NZ	CE
3	4	HH2	CH2	NZ	CE
3	4	HH3	CH3	NZ	CE

